In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from sliced_wasserstein import sliced_wasserstein_distance
from c2st import c2st_knn, c2st_nn, c2st_rf

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/frequency_power_analysis/frequency_power_data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,29,34,42,43,52,60,62,67,69,80]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:
def amplify_frequency_band(signal, sampling_rate, low_freq, high_freq, amplification_factor):
    # FFT
    freqs = np.fft.rfftfreq(len(signal), d=1/sampling_rate)
    fft_coeffs = np.fft.rfft(signal)

    # Amplify the specified frequency band
    band_mask = (freqs >= low_freq) & (freqs <= high_freq)
    fft_coeffs[band_mask] *= amplification_factor

    # Inverse FFT
    modified_signal = np.fft.irfft(fft_coeffs, n=len(signal))
    return modified_signal

In [ ]:
def load_model(cfg, start_index=100, subject_index=2):
    save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/frequency_power_analysis/finetune_model_weights"
    
    file_path = os.path.join(save_path, f"subject_{subject_index}", f"model_checkpoint_pretrain_subject_index_{subject_index}_start_idx_{start_index}.pth") 
    trunk_net = TrunkNet(n_chans=input_shape_st[0], n_times=input_shape_st[1])
    head_net = HeadNet(64, 1)  # Assuming these are the correct dimensions
    model = S4PatchedFinalNet(64, trunk_net, head_net)
    
    weights = torch.load(file_path)
    model.load_state_dict(weights)
    #model.load_state_dict(full_checkpoint['model_state_dict'])
    model.eval()
    model.to(device)
    return model

In [ ]:
cfg = load_config()

subject_index = 2
cfg.dataset.subject_index = subject_index
cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
cli_args = parse_args()
cfg = update_config(cfg, cli_args)
save_config(cfg)
all_epochs, labels_raw, _, _, _, _, _, ch_names = load_eeg_data(cfg)
all_epochs = all_epochs[150:]
labels_raw = labels_raw[150:]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

In [ ]:
freq_bands = {
              #"delta": (0, 4),
            "theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}
amplification_factors = [0.2,0.5,2,3,5,10]

In [ ]:

perturbed_prediction_dict = {}

for band_name, (low_freq, high_freq) in freq_bands.items():
    print(band_name)
    for factor in amplification_factors:
        perturbed_channel_wise = {}
        # only for a given channel change the power in the given frequency band (but do it for all samples)
        for ch_idx, ch_name in enumerate(ch_names):
            pred_label_perturbed = np.zeros((all_epochs.shape[0]))
            uncertainties_perturbed = np.zeros((all_epochs.shape[0]))
            for sample in range(all_epochs.shape[0]):
                perturbed_sample = copy.deepcopy(all_epochs[sample])
                start_index = sample+100
                perturbed_sample[ch_idx] = amplify_frequency_band(all_epochs[sample, ch_idx], 1000, low_freq, high_freq, factor)
                inputs = torch.from_numpy(perturbed_sample)
                inputs = inputs.to(device).float()
                inputs = inputs.unsqueeze(0)
                model = load_model(cfg,start_index=start_index, subject_index=subject_index)
                pred_mean, log_var = model(inputs)[:, 0], model(inputs)[:, 1]
                var = torch.exp(log_var)
                pred_label_perturbed[sample] = pred_mean.cpu().detach().numpy()
                uncertainties_perturbed[sample] = var.cpu().detach().numpy()

            perturbed_channel_wise[ch_name] = (pred_label_perturbed, uncertainties_perturbed)
        perturbed_prediction_dict[f"band_{band_name}_factor_{factor}"] = perturbed_channel_wise
        np.save(f"perturbed_prediction_dict_{band_name}_factor_{factor}_subject{SUBJECT}.npy", perturbed_channel_wise)

            

In [ ]:
cfg = load_config()
pred_label_original = np.zeros((all_epochs.shape[0]))
uncertainties_original = np.zeros((all_epochs.shape[0]))
input_shape_st = (60, 900)                
for i in tqdm(range(0, len(all_epochs))):
    start_index = i+100
    inputs = torch.from_numpy(all_epochs[i])
    inputs = inputs.to(device).float()
    inputs = inputs.unsqueeze(0)

    model = load_model(cfg, start_index=start_index, subject_index=subject_index)
    pred_mean, log_var = model(inputs)[:, 0], model(inputs)[:, 1]
    var = torch.exp(log_var)
    pred_label_original[i] = pred_mean.cpu().detach().numpy()
    uncertainties_original[i] = var.cpu().detach().numpy()


# Difference in prediction per individual pertubred channel between original prediction and prediction on perturbed sample

We have predicted amplitudes for changing the power of 

right now we would have a single plot per frequency band per amp factor and per channel making 5x6x60 plots. Looking at all of them separately seems unreasonable. How to aggregate information across channels? 

Goal: Find out changing power in which channels changes the prediction the most.

1. Reduce channel dim by only plotting mean/median amplitude per channel (Problem: Different trials might behave differently to change in power. How to analyse this?)
2. make bar-plot of mean/median change in amplitude per channel for each freq band and amp factor (original amplitude - perturbed amplitude)


In [ ]:

def calculate_diff_per_channel(pred_label_original, freq_bands, amplification_factors, ch_names):
    mean_diff_per_channel = {}
    median_diff_per_channel = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diff_per_channel[band_name] = {}
        median_diff_per_channel[band_name] = {}
        for factor in amplification_factors:
            perturbed_data = np.load(f"channel_wise/perturbed_prediction_dict_{band_name}_factor_{factor}.npy", allow_pickle=True).item()
            mean_diff_per_channel[band_name][factor] = {}
            median_diff_per_channel[band_name][factor] = {}
            for ch_name in ch_names:
                perturbed_amplitude = perturbed_data[ch_name][0]
                diff = np.abs(pred_label_original - perturbed_amplitude)
                mean_diff_per_channel[band_name][factor][ch_name] = np.mean(diff)
                median_diff_per_channel[band_name][factor][ch_name] = np.median(diff)
    
    return mean_diff_per_channel, median_diff_per_channel

mean_diff_per_channel, median_diff_per_channel = calculate_diff_per_channel(pred_label_original, freq_bands, amplification_factors, ch_names)

fig, axes = plt.subplots( len(amplification_factors), len(freq_bands), figsize=(20, 15), sharex=True)
fig.suptitle('Mean/Median Amplitude Difference per Channel')
fig.tight_layout()
for i, factor in enumerate(amplification_factors):
    for j, (band_name, _) in enumerate(freq_bands.items()):

        ax = axes[i, j]
        mean_diffs = [mean_diff_per_channel[band_name][factor][ch] for ch in ch_names]
        median_diffs = [median_diff_per_channel[band_name][factor][ch] for ch in ch_names]
        ax.scatter(ch_names, mean_diffs, label='Mean', color='blue')
        ax.scatter(ch_names, median_diffs, label='Median', color='red')
        #ax.plot(np.arange(len(ch_names)), [0]*len(ch_names), "k--")
        ax.set_title(f'{band_name} - Factor {factor}')
        ax.set_xlabel('Channel')
        ax.set_ylabel('Amplitude Difference')
        ax.legend()
        ax.tick_params(axis='x', rotation=90)




this plot already seems to suggest a lot of outliers bein present given that the difference between mean and median is that big. Next is finding out which top-k channels lead to the largest absolute mean/median difference in amplitude between the perturbed samples and the original samples

## top 10 channels with largest prediction differenceper frequency band

In [ ]:
def get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors, top_k=10):
    top_channels_mean = {}
    top_channels_median = {}
    prediction_diff_mean = {}
    prediction_diff_median = {}
    
    for band_name in freq_bands.keys():
        top_channels_mean[band_name] = {}
        top_channels_median[band_name] = {}
        prediction_diff_mean[band_name] = {}
        prediction_diff_median[band_name] = {}
        
        for factor in amplification_factors:
            mean_diffs = mean_diff_per_channel[band_name][factor]
            median_diffs = median_diff_per_channel[band_name][factor]
            
            # Sort the channels based on their differences
            sorted_mean_diffs = sorted(mean_diffs.items(), key=lambda item: item[1], reverse=True)
            sorted_median_diffs = sorted(median_diffs.items(), key=lambda item: item[1], reverse=True)
            
            # Select the top k channels
            top_channels_mean[band_name][factor] = [ch for ch, _ in sorted_mean_diffs[:top_k]]
            top_channels_median[band_name][factor] = [ch for ch, _ in sorted_median_diffs[:top_k]]
            
            # Store the prediction differences for the top k channels
            prediction_diff_mean[band_name][factor] = {ch: mean_diffs[ch] for ch in top_channels_mean[band_name][factor]}
            prediction_diff_median[band_name][factor] = {ch: median_diffs[ch] for ch in top_channels_median[band_name][factor]}
    
    return top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median

top_channels_mean, top_channels_median, prediction_diff_mean, prediction_diff_median = get_top_channels(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors)




In [ ]:
top_channels_mean["gamma"], top_channels_median["gamma"]

The agreement with the top-k channels seems to be very high at first glance. Also notice that the order of channels tends to differ slightly differ

In [ ]:
top_channels_mean["alpha"], top_channels_median["alpha"]

In [ ]:
top_channels_mean["theta"], top_channels_median["theta"]

In [ ]:
top_channels_mean["beta"], top_channels_median["beta"]

check how much "importances" agree with each other and also how much they agree with the top-k returned by the explainability method. Also note that the importances here are display in descending order of "importance"

"importance" - magnitude of the absolute difference.

might be interesting to check if there is any difference between prediction with low amplitude and predictions with high amplitude (boolean pooling)

## top channel per amplification factor

In [ ]:
biggest_diff_channels = {}

for factor in amplification_factors:
    mean_diffs_agg = {ch: 0 for ch in ch_names}
    median_diffs_agg = {ch: 0 for ch in ch_names}
    
    for band_name in freq_bands.keys():
        mean_diffs = mean_diff_per_channel[band_name][factor]
        median_diffs = median_diff_per_channel[band_name][factor]
        
        for ch in ch_names:
            mean_diffs_agg[ch] += mean_diffs[ch]
            median_diffs_agg[ch] += median_diffs[ch]
    
    # Find the channel with the maximum aggregated mean difference
    max_mean_channel = max(mean_diffs_agg, key=mean_diffs_agg.get)
    max_mean_value = mean_diffs_agg[max_mean_channel]
    
    # Find the channel with the maximum aggregated median difference
    max_median_channel = max(median_diffs_agg, key=median_diffs_agg.get)
    max_median_value = median_diffs_agg[max_median_channel]
    
    biggest_diff_channels[factor] = {
        'max_mean_channel': max_mean_channel,
        'max_mean_value': max_mean_value,
        'max_median_channel': max_median_channel,
        'max_median_value': max_median_value
    }

# Print the results
for factor, diff_info in biggest_diff_channels.items():
    print(f"Amplification Factor: {factor}")
    print(f"  Max Mean Channel: {diff_info['max_mean_channel']} (Value: {diff_info['max_mean_value']})")
    print(f"  Max Median Channel: {diff_info['max_median_channel']} (Value: {diff_info['max_median_value']})")

## agreement between top-10 channels per prediction difference and top-10 channels to interpretability method (gradShap)

In [ ]:
top_k = np.load("top_k_dict.npy", allow_pickle=True).item()

### for median difference

In [ ]:
# Extract the top 10 channels from top_k[2]
top_channels_top_k_2 = top_k[2]

# Initialize a dictionary to store the agreement results
agreement_results = {}

# Iterate over each frequency band
for band_name in freq_bands.keys():
    agreement_results[band_name] = {}
    for factor in amplification_factors:
        # Extract the top 10 channels from top_channels_median for the current band and factor
        top_channels_median_band_factor = top_channels_median[band_name][factor]
        
        # Check the number of top channels that agree
        agreement_count = sum(1 for ch in top_channels_top_k_2 if ch in top_channels_median_band_factor)
        
        # Calculate the agreement ratio
        agreement_ratio = agreement_count / 10.0
        
        # Store the result
        agreement_results[band_name][factor] = agreement_ratio

# Print the agreement results
for band_name, factors in agreement_results.items():
    print(f"Frequency Band: {band_name}")
    for factor, agreement_ratio in factors.items():
        print(f"  Amplification Factor: {factor} - Agreement Ratio: {agreement_ratio:.2f}")

# Prepare data for plotting
bands = list(agreement_results.keys())
factors = list(agreement_results[bands[0]].keys())
agreement_matrix = np.zeros((len(bands), len(factors)))

for i, band in enumerate(bands):
    for j, factor in enumerate(factors):
        agreement_matrix[i, j] = agreement_results[band][factor]

# Plot the agreement matrix
fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.matshow(agreement_matrix, cmap='bwr')

# Add color bar

#fig.colorbar(cax,fraction=0.035, pad=0.04)

# Set axis labels
ax.set_xticks(np.arange(len(factors)))
ax.set_yticks(np.arange(len(bands)))
ax.set_xticklabels(factors)
ax.set_yticklabels(bands)

# Rotate the tick labels and set their alignment
plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")

# Add text annotations
for i in range(len(bands)):
    for j in range(len(factors)):
        text = ax.text(j, i, f"{agreement_matrix[i, j]:.2f}", ha="center", va="center", color="white")

ax.set_xlabel('Amplification Factors')
ax.set_ylabel('Frequency Bands')
ax.set_title('Agreement ratio of Top 10 channels of prediction difference and explanation function')
fig.savefig(f'agreement_matrix_top_k_Difference_and_XAI_subject.png')
plt.show()


### for mean difference

In [ ]:
# Initialize a dictionary to store the agreement results for top_channels_mean
agreement_results_mean = {}

# Iterate over each frequency band
for band_name in freq_bands.keys():
    agreement_results_mean[band_name] = {}
    for factor in amplification_factors:
        # Extract the top 10 channels from top_channels_mean for the current band and factor
        top_channels_mean_band_factor = top_channels_mean[band_name][factor]
                
        # Check the number of top channels that agree
        agreement_count = sum(1 for ch in top_channels_top_k_2 if ch in top_channels_mean_band_factor)
                
        # Calculate the agreement ratio
        agreement_ratio = agreement_count / 10.0
                
        # Store the result
        agreement_results_mean[band_name][factor] = agreement_ratio

# Print the agreement results for top_channels_mean
for band_name, factors in agreement_results_mean.items():
    print(f"Frequency Band: {band_name}")
    for factor, agreement_ratio in factors.items():
        print(f" Amplification Factor: {factor} - Agreement Ratio: {agreement_ratio:.2f}")

Note that doing this works best for gamma and beta bands. Only thing left to check is how high agreement is between frequency bands themselves and how all frequency bands combined agree with the top-k

# Agreement of top-10 channlels per frequency band(How much to top channels per prediction difference agree across frequency bands)

In [ ]:
agreement_between_bands = {}

# Iterate over each pair of frequency bands
for band_name_1 in freq_bands.keys():
    agreement_between_bands[band_name_1] = {}
    for band_name_2 in freq_bands.keys():
        if band_name_1 != band_name_2:
            top_channels_1 = set(top_channels_median[band_name_1][2])
            top_channels_2 = set(top_channels_median[band_name_2][2])
            
            # Calculate the intersection and agreement ratio
            intersection = top_channels_1.intersection(top_channels_2)
            agreement_ratio = len(intersection) / len(top_channels_1)
            
            # Store the result
            agreement_between_bands[band_name_1][band_name_2] = agreement_ratio

# Print the agreement results
for band_name_1, bands in agreement_between_bands.items():
    print(f"Frequency Band: {band_name_1}")
    for band_name_2, agreement_ratio in bands.items():
        print(f"  Compared with Band: {band_name_2} - Agreement Ratio: {agreement_ratio:.2f}")

# Prepare data for plotting
bands = list(agreement_between_bands.keys())
agreement_matrix = np.zeros((len(bands), len(bands)))

for i, band_1 in enumerate(bands):
    for j, band_2 in enumerate(bands):
        if band_1 != band_2:
            agreement_matrix[i, j] = agreement_between_bands[band_1][band_2]

# Plot the agreement matrix
fig, ax = plt.subplots(figsize=(10, 8))
cax = ax.matshow(agreement_matrix, cmap='viridis')

# Add color bar
fig.colorbar(cax)

# Set axis labels
ax.set_xticks(np.arange(len(bands)))
ax.set_yticks(np.arange(len(bands)))
ax.set_xticklabels(bands)
ax.set_yticklabels(bands)

# Rotate the tick labels and set their alignment
plt.setp(ax.get_xticklabels(), rotation=45, ha="left", rotation_mode="anchor")

# Add text annotations
for i in range(len(bands)):
    for j in range(len(bands)):
        if i != j:
            text = ax.text(j, i, f"{agreement_matrix[i, j]:.2f}", ha="center", va="center", color="white")

ax.set_xlabel('Frequency Bands')
ax.set_ylabel('Frequency Bands')
ax.set_title('Agreement Between Top 10 Channels of Different Frequency Bands')
fig.savefig(f"agreement_matrix_across_channel_differences.png")
plt.show()


Agreement is (unsurprisinlgy) higher for frequency bands that are closer together.

In [ ]:

unique_channels = set()

for band_name in top_channels_median.keys():
    unique_channels.update(top_channels_median[band_name][2])

print(f"Number of unique channels: {len(unique_channels)}")
print(f"Unique channels: {unique_channels}")


again get a rank correlation between differences of prediction per frequency band importance uncovered by interpretability method?

In [ ]:
top_k_60 = np.load("top_k_per_subject_60.npy", allow_pickle=True).item()
top_k_60_abs = np.load("top_k_per_subject_60_abs.npy", allow_pickle=True).item()

In [ ]:
# Extract the top_k_60 for subject 2
top_k_60_subject_2 = top_k_60[2]

# Sort the dictionary by values in descending order
sorted_top_k_60_subject_2 = dict(sorted(top_k_60_subject_2.items(), key=lambda item: item[1], reverse=True))

# Print the sorted dictionary
print(sorted_top_k_60_subject_2)

In [ ]:

def get_prediction_diff(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors):
    prediction_diff_mean = {}
    prediction_diff_median = {}
    
    for band_name in freq_bands.keys():
        prediction_diff_mean[band_name] = {}
        prediction_diff_median[band_name] = {}
        
        for factor in amplification_factors:
            mean_diffs = mean_diff_per_channel[band_name][factor]
            median_diffs = median_diff_per_channel[band_name][factor]
            
            # Store the prediction differences without sorting
            prediction_diff_mean[band_name][factor] = mean_diffs
            prediction_diff_median[band_name][factor] = median_diffs
    
    return prediction_diff_mean, prediction_diff_median

prediction_diff_mean_unsorted, prediction_diff_median_unsorted = get_prediction_diff(mean_diff_per_channel, median_diff_per_channel, freq_bands, amplification_factors)

In [ ]:
from scipy.stats import pearsonr

# Extract the values of top_k_60 for subject 2
top_k_60_values = list(top_k_60[2].values())

# Initialize a dictionary to store the correlation results
correlation_results = {}

# Iterate over each frequency band
for band_name in freq_bands.keys():
    # Extract the values of prediction_diff_mean_unsorted for the current band and factor 2
    prediction_diff_values = list(prediction_diff_mean_unsorted[band_name][2].values())
    
    # Compute the Pearson rank correlation
    correlation, p_val = pearsonr(top_k_60_values, prediction_diff_values)
    
    # Store the result
    correlation_results[band_name] = (correlation, p_val)

# Print the correlation results
for band_name, (correlation, pval) in correlation_results.items():
    print(f"Frequency Band: {band_name} - Pearson Rank Correlation: {correlation:.3f}, p-value: {pval:.3f}")

    

In [ ]:
from scipy.stats import pearsonr

# Extract the values of top_k_60 for subject 2
top_k_60_values = list(top_k_60[2].values())

# Initialize a dictionary to store the correlation results
correlation_results = {}

# Iterate over each frequency band
for band_name in freq_bands.keys():
    # Extract the values of prediction_diff_mean_unsorted for the current band and factor 2
    prediction_diff_values = list(prediction_diff_median_unsorted[band_name][2].values())
    
    # Compute the Pearson rank correlation
    correlation, p_val = pearsonr(top_k_60_values, prediction_diff_values)
    
    # Store the result
    correlation_results[band_name] = (correlation, p_val)

# Print the correlation results
for band_name, (correlation, pval) in correlation_results.items():
    print(f"Frequency Band: {band_name} - Pearson Rank Correlation: {correlation:.3f}, p-value: {pval:.3f}")

for the beta and gamma band these rank correlations are not only significant but also quite high. This further provided evidence as the gamma and beta band being the most important for the predicted amplitude, which is in line with the results of frequency ROAR

Furthermore taking the median difference seems to increase rank correlations between the interpretability method and the difference in predicted amplitude

## for top_k abs

In [ ]:
top_k_abs = np.load("top_k_abs.npy", allow_pickle=True).item()
top_k_abs[2]

In [ ]:
# Extract the top 10 channels from top_k[2]
top_channels_top_k_2 = top_k_abs[2]

# Initialize a dictionary to store the agreement results
agreement_results = {}

# Iterate over each frequency band
for band_name in freq_bands.keys():
    agreement_results[band_name] = {}
    for factor in amplification_factors:
        # Extract the top 10 channels from top_channels_median for the current band and factor
        top_channels_median_band_factor = top_channels_median[band_name][factor]
        
        # Check the number of top channels that agree
        agreement_count = sum(1 for ch in top_channels_top_k_2 if ch in top_channels_median_band_factor)
        
        # Calculate the agreement ratio
        agreement_ratio = agreement_count / 10.0
        
        # Store the result
        agreement_results[band_name][factor] = agreement_ratio

# Print the agreement results
for band_name, factors in agreement_results.items():
    print(f"Frequency Band: {band_name}")
    for factor, agreement_ratio in factors.items():
        print(f"  Amplification Factor: {factor} - Agreement Ratio: {agreement_ratio:.2f}")

for top_k_abs interestingly the agreement seems to drop for all frequency bands except for gamma, where a slightly higher agreement (about 10% so 1 channel) can be observed

In [ ]:
# Extract the top_k_60_abs for subject 2
top_k_60_subject_2 = top_k_60_abs[2]

# Sort the dictionary by values in descending order
sorted_top_k_60_subject_2 = dict(sorted(top_k_60_subject_2.items(), key=lambda item: item[1], reverse=True))

# Print the sorted dictionary
print(sorted_top_k_60_subject_2)

In [ ]:
from scipy.stats import pearsonr

# Extract the values of top_k_60 for subject 2
top_k_60_values = list(top_k_60_abs[2].values())

# Initialize a dictionary to store the correlation results
correlation_results = {}

# Iterate over each frequency band
for band_name in freq_bands.keys():
    # Extract the values of prediction_diff_mean_unsorted for the current band and factor 2
    prediction_diff_values = list(prediction_diff_mean_unsorted[band_name][2].values())
    
    # Compute the Pearson rank correlation
    correlation, p_val = pearsonr(top_k_60_values, prediction_diff_values)
    
    # Store the result
    correlation_results[band_name] = (correlation, p_val)

# Print the correlation results
for band_name, (correlation, pval) in correlation_results.items():
    print(f"Frequency Band: {band_name} - Pearson Rank Correlation: {correlation:.3f}, p-value: {pval:.3f}")

# Prepare data for plotting
bands = list(correlation_results.keys())
correlations = [correlation_results[band][0] for band in bands]
p_values = [correlation_results[band][1] for band in bands]

# Plot the correlation results
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(bands, correlations, color='skyblue', alpha=0.7)
ax.set_xlabel('Frequency Bands')
ax.set_ylabel('Pearson Rank Correlation')
ax.set_title('Pearson Rank Correlation of all channels between explanation function and feature')


# Add text annotations for p-values
for i, p_val in enumerate(p_values):
    ax.text(i, correlations[i] + 0.02, f'p={p_val:.3f}', ha='center', va='bottom', fontsize=10)
plt.savefig(f"Rank_correlation_top_k_abs_subject_{2}.png")
plt.show()

In [ ]:
from scipy.stats import pearsonr

# Extract the values of top_k_60 for subject 2
top_k_60_values = list(top_k_60_abs[2].values())

# Initialize a dictionary to store the correlation results
correlation_results = {}

# Iterate over each frequency band
for band_name in freq_bands.keys():
    # Extract the values of prediction_diff_mean_unsorted for the current band and factor 2
    prediction_diff_values = list(prediction_diff_median_unsorted[band_name][2].values())
    
    # Compute the Pearson rank correlation
    correlation, p_val = pearsonr(top_k_60_values, prediction_diff_values)
    
    # Store the result
    correlation_results[band_name] = (correlation, p_val)

# Print the correlation results
for band_name, (correlation, pval) in correlation_results.items():
    print(f"Frequency Band: {band_name} - Pearson Rank Correlation: {correlation:.3f}, p-value: {pval:.3f}")



# Prepare data for plotting
bands = list(correlation_results.keys())
correlations = [correlation_results[band][0] for band in bands]
p_values = [correlation_results[band][1] for band in bands]

# Plot the correlation results
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(bands, correlations, color='skyblue', alpha=0.7)
ax.set_xlabel('Frequency Bands')
ax.set_ylabel('Pearson Rank Correlation')
ax.set_title('Pearson Rank Correlation of all channels between explanation function and prediction difference')

# Add text annotations for p-values
for i, p_val in enumerate(p_values):
    ax.text(i, correlations[i] + 0.02, f'p={p_val:.3f}', ha='center', va='bottom', fontsize=10)

# Remove the top and right spines (black box)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.savefig(f"Rank_correlation_top_k_abs_median_difference_subject_{2}.png")
plt.show()

taking the top_60_abs values of the interpretability method increases rank correlation by a significant amount. This is as expected as very negative and very positive values of the interpretabiliy method suggest high importance in a regression task. Low values means this feature will decrease the predicted amplitude and high values means this feature is important for increasing the predicted ampitude, but BOTH are informative about the predited amplitude, which is what we are looking for

The results above increase the evidence that beta and gamma bands are the most important and that these are also what is attended to the most by XAI attribution methods

# PDP and ICE for each channel and frequency band separately

## PDP

In [ ]:
top_channels_median["gamma"]

In [ ]:
def plot_PDP_per_channel_and_freq_band(pred_label_original, freq_bands, amplification_factors, ch_names, abs_diff=False):
    mean_diff_per_channel = {}
    median_diff_per_channel = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diff_per_channel[band_name] = {}
        median_diff_per_channel[band_name] = {}
        fig, axs = plt.subplots(ncols=3, nrows=20, figsize=(20, 40), sharex=True)
        fig.tight_layout()
        fig.suptitle(f'Amplitude Difference per Channel for {band_name}')
        for ch_idx,ch_name in enumerate(ch_names):
            mean_diffs = []
            median_diffs = []
            for factor_idx,factor in enumerate(amplification_factors):
                perturbed_data = np.load(f"channel_wise/perturbed_prediction_dict_{band_name}_factor_{factor}_SUBJECT_{2}.npy", allow_pickle=True).item()
                perturbed_amplitude = perturbed_data[ch_name][0]
                if abs_diff:
                    diff = np.abs(pred_label_original - perturbed_amplitude)
                else:
                    diff = pred_label_original - perturbed_amplitude
                mean_diffs.append(np.mean(diff))
                #median_diffs.append(np.median(diff))
            
            axs[ch_idx//3, ch_idx%3].plot(amplification_factors, mean_diffs, label='Mean', color='blue')
            axs[ch_idx//3, ch_idx%3].set_xticks(amplification_factors)
            axs[ch_idx//3, ch_idx%3].set_xticklabels(amplification_factors)
            #ax.scatter(amplification_factors, median_diffs, label='Median', color='red')
            axs[ch_idx//3, ch_idx%3].set_title(f'{ch_name}')
            if ch_idx%3 >=57:
                axs[ch_idx//3, ch_idx%3].set_xlabel('Amplification Factor')
            if ch_idx%3 == 0:
                if abs_diff:
                    axs[ch_idx//3, ch_idx%3].set_ylabel('Absolute Amplitude Difference')
                else:
                    axs[ch_idx//3, ch_idx%3].set_ylabel('Amplitude Difference')
            
            #ax.legend()
           

def plot_top_10_channels(pred_label_original, freq_bands, amplification_factors, ch_names, top_channels_median, abs_diff=False):
    fig, axs = plt.subplots(ncols=2, nrows=5, figsize=(13, 12), sharex=True, gridspec_kw={'hspace': 0.3})
    #fig.tight_layout()
    fig.suptitle('Top 10 Channels with Largest Amplitude Difference')
    
    for band_name, (low_freq, high_freq) in freq_bands.items():
        top_10_channels = top_channels_median[band_name][2]  # Assuming factor 2 is used to determine top 10 channels
        for ch_idx, ch_name in enumerate(top_10_channels):
            mean_diffs = []
            for factor in amplification_factors:
                perturbed_data = np.load(f"channel_wise/perturbed_prediction_dict_{band_name}_factor_{factor}.npy", allow_pickle=True).item()
                perturbed_amplitude = perturbed_data[ch_name][0]
                if abs_diff:
                    diff = np.abs(pred_label_original - perturbed_amplitude)
                else:
                    diff = pred_label_original - perturbed_amplitude
                mean_diffs.append(np.mean(diff))
                    
            axs[ch_idx//2, ch_idx%2].plot(amplification_factors, mean_diffs, label=f'{band_name} Band')
            axs[ch_idx//2, ch_idx%2].set_xticks(amplification_factors)
            axs[ch_idx//2, ch_idx%2].set_xticklabels(amplification_factors)
            axs[ch_idx//2, ch_idx%2].set_title(f'{ch_name}')
            if ch_idx%2 == 0:
                if abs_diff:
                    axs[ch_idx//2, ch_idx%2].set_ylabel('Absolute Amplitude Difference')
                else:
                    axs[ch_idx//2, ch_idx%2].set_ylabel('Amplitude Difference')
            if ch_idx//2 == 4:
                axs[ch_idx//2, ch_idx%2].set_xlabel('Amplification Factor')
    
    for ax in axs.flat:
        ax.legend()
    fig.savefig(f"top_10_channels_amplitude_difference_behavior.png")
    plt.show()
    

plot_top_10_channels(pred_label_original, freq_bands, amplification_factors, ch_names, top_channels_median, abs_diff=False)


In [ ]:
plot_PDP_per_channel_and_freq_band(pred_label_original, freq_bands, amplification_factors, ch_names)

In [ ]:
plot_PDP_per_channel_and_freq_band(pred_label_original, freq_bands, amplification_factors, ch_names, abs_diff=True)

Notice: No matter the channel and not matter the frequency band, the predicted amplitude seems to increase with and increase in power in the band (if we take the absolute difference of the predicted amplitudes)

This suggests what?

## ICE

In [ ]:
def plot_ICE_per_channel_and_freq_band(pred_label_original, freq_bands, amplification_factors, ch_names, abs_diff=False):
    mean_diff_per_channel = {}
    median_diff_per_channel = {}
    ICE_array = np.zeros((len(ch_names), len(amplification_factors), len(pred_label_original)))
    for band_name, (low_freq, high_freq) in freq_bands.items():
        mean_diff_per_channel[band_name] = {}
        median_diff_per_channel[band_name] = {}
        fig, axs = plt.subplots(ncols=3, nrows=20, figsize=(20, 40), sharex=True)
        fig.tight_layout()
        fig.suptitle(f'Amplitude Difference per Channel for {band_name}')
        for ch_idx,ch_name in enumerate(ch_names):
            mean_diffs = []
            median_diffs = []
            for factor_idx, factor in enumerate(amplification_factors):
                perturbed_data = np.load(f"channel_wise/perturbed_prediction_dict_{band_name}_factor_{factor}.npy", allow_pickle=True).item()
                perturbed_amplitude = perturbed_data[ch_name][0]
                if abs_diff:
                    diff = np.abs(pred_label_original - perturbed_amplitude)
                else: 
                    diff = pred_label_original - perturbed_amplitude
        
                mean_diffs.append(np.mean(diff))
                ICE_array[ch_idx, factor_idx] = diff
        
            axs[ch_idx//3, ch_idx%3].plot(amplification_factors, ICE_array[ch_idx], label='Mean', color='blue', alpha=0.35)
            axs[ch_idx//3, ch_idx%3].plot(amplification_factors, mean_diffs, label='Mean', color='black')
            axs[ch_idx//3, ch_idx%3].set_xticks(amplification_factors)
            axs[ch_idx//3, ch_idx%3].set_xticklabels(amplification_factors)
            axs[ch_idx//3, ch_idx%3].set_title(f'{ch_name}')
            if ch_idx%3 >= 57:
                axs[ch_idx//3, ch_idx%3].set_xlabel('Amplification Factor')
            if ch_idx%3 == 0:
                if abs_diff:
                    axs[ch_idx//3, ch_idx%3].set_ylabel('Absolute Amplitude Difference')
                else:
                    axs[ch_idx//3, ch_idx%3].set_ylabel('Amplitude Difference')



def plot_top_10_ICE(pred_label_original, freq_bands, amplification_factors, ch_names, top_channels_median, abs_diff=False):
    for band_name, (low_freq, high_freq) in freq_bands.items():
        fig, axs = plt.subplots(ncols=2, nrows=5, figsize=(15, 12), sharex=True)
        fig.tight_layout()
        fig.suptitle(f'ICE for Top 10 Channels - {band_name} Band')
        top_10_channels = top_channels_median[band_name][2]  # Assuming factor 2 is used to determine top 10 channels
        ICE_array = np.zeros((len(top_10_channels), len(amplification_factors), len(pred_label_original)))
        for ch_idx, ch_name in enumerate(top_10_channels):
            for factor_idx, factor in enumerate(amplification_factors):
                perturbed_data = np.load(f"channel_wise/perturbed_prediction_dict_{band_name}_factor_{factor}.npy", allow_pickle=True).item()
                perturbed_amplitude = perturbed_data[ch_name][0]
                if abs_diff:
                    diff = np.abs(pred_label_original - perturbed_amplitude)
                else:
                    diff = pred_label_original - perturbed_amplitude
                ICE_array[ch_idx, factor_idx] = diff

            axs[ch_idx//2, ch_idx%2].plot(amplification_factors, ICE_array[ch_idx], color='blue', alpha=0.35)
            axs[ch_idx//2, ch_idx%2].plot(amplification_factors, ICE_array[ch_idx].mean(axis=1), color='black')
            axs[ch_idx//2, ch_idx%2].set_xticks(amplification_factors)
            axs[ch_idx//2, ch_idx%2].set_xticklabels(amplification_factors)
            axs[ch_idx//2, ch_idx%2].set_title(f'{ch_name}')
            if ch_idx%2 == 0:
                if abs_diff:
                    axs[ch_idx//2, ch_idx%2].set_ylabel('Absolute Amplitude Difference')
                else:
                    axs[ch_idx//2, ch_idx%2].set_ylabel('Amplitude Difference')
            if ch_idx//2 == 4:
                axs[ch_idx//2, ch_idx%2].set_xlabel('Amplification Factor')
        fig.savefig(f"ICE_top_10_channels_{band_name}.png")

                         
plot_top_10_ICE(pred_label_original, freq_bands, amplification_factors, ch_names, top_channels_median, abs_diff=False)


Take away message: Different trials react differently to a change in amplitude in a channel, Maybe we ca get groups of trials this way?

In [ ]:
plot_ICE_per_channel_and_freq_band(pred_label_original, freq_bands, amplification_factors, ch_names)

In [ ]:
plot_top_10_channels_ICE(pred_label_original, freq_bands, amplification_factors, ch_names, top_channels_median, abs_diff=False)

for the normal differences of original_prediction - perturbed_prediction there might be groups of trials that behave differently from each other

In [ ]:
plot_ICE_per_channel_and_freq_band(pred_label_original, freq_bands, amplification_factors, ch_names, abs_diff=True )

check if the most important channels have a greater variance in ICEs compared to less important channels